# Arabic Sign Language Translator — Inference & Evaluation

This notebook loads the pre-trained model and runs predictions.

**Requirements:** `pip install tensorflow opencv-python matplotlib`  
**Model file:** Place `SL.h5` inside the `model/` folder before running.

No dataset or training required.

## 1. Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from keras.models import load_model

## 2. Class Labels

In [ ]:
LABELS = {
    0: 'أ',  1: 'ب',  2: 'ت',  3: 'ث',  4: 'ج',
    5: 'ح',  6: 'خ',  7: 'د',  8: 'ذ',  9: 'ر',
    10: 'ز', 11: 'س', 12: 'ش', 13: 'ص', 14: 'ض',
    15: 'ط', 16: 'ظ', 17: 'ع', 18: 'غ', 19: 'ف',
    20: 'ق', 21: 'ك', 22: 'ل', 23: 'م', 24: 'ن',
    25: 'ه', 26: 'و', 27: 'ي', 28: 'ة', 29: 'ال',
    30: 'لا', 31: 'ئ'
}

## 3. Load Model

In [ ]:
MODEL_PATH = 'model/SL.h5'
model = load_model(MODEL_PATH)
model.summary()

## 4. Predict on a Single Image

In [ ]:
def preprocess_image(image_path):
    """Load and preprocess a single image for model input."""
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, (64, 64))
    img_input = img_resized.reshape(1, 64, 64, 3).astype('float32') / 255.0
    return img, img_input


def predict_sign(model, image_path):
    """Return predicted label and confidence for a given image."""
    img, img_input = preprocess_image(image_path)
    prediction = model.predict(img_input, verbose=0)
    class_idx = np.argmax(prediction)
    confidence = prediction[0][class_idx]
    label = LABELS[class_idx]
    return label, confidence, img


# --- Example usage ---
# Replace with your own test image path
TEST_IMAGE = 'results/sample_test.jpg'

if os.path.exists(TEST_IMAGE):
    label, confidence, img = predict_sign(model, TEST_IMAGE)
    print(f"Predicted: {label}  (confidence: {confidence:.2%})")
else:
    print(f"No test image found at '{TEST_IMAGE}'. Update TEST_IMAGE to a real path.")

## 5. Batch Prediction & Visualization

In [ ]:
def visualize_predictions(model, image_paths, labels_map, cols=4):
    """
    Run predictions on a list of image paths and display a grid.
    Saves output to results/predictions_grid.png.
    """
    rows = (len(image_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    for i, path in enumerate(image_paths):
        label, confidence, img = predict_sign(model, path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(f"{label}  ({confidence:.0%})", fontsize=14)
        axes[i].axis('off')

    # Hide any empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Arabic Sign Language — Predictions', fontsize=16, y=1.02)
    plt.tight_layout()

    os.makedirs('results', exist_ok=True)
    plt.savefig('results/predictions_grid.png', dpi=150, bbox_inches='tight')
    print("Saved: results/predictions_grid.png")
    plt.show()


# --- Example: run on a folder of test images ---
# test_folder = 'path/to/your/test/images'
# image_paths = [os.path.join(test_folder, f) for f in os.listdir(test_folder) if f.endswith('.jpg')]
# visualize_predictions(model, image_paths[:8], LABELS)

## 6. Live Webcam Demo

Run this cell to activate your webcam. Press **`q`** to quit.

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Could not open webcam.")
else:
    print("Webcam open. Press 'q' to quit.")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Preprocess and predict
        img_input = cv2.resize(frame, (64, 64)).reshape(1, 64, 64, 3).astype('float32') / 255.0
        prediction = model.predict(img_input, verbose=0)
        class_idx = np.argmax(prediction)
        label = LABELS[class_idx]
        confidence = prediction[0][class_idx]

        # Overlay prediction on frame
        cv2.putText(
            frame,
            f"{label}  {confidence:.0%}",
            (30, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            2.5,
            (0, 0, 255),
            5,
            cv2.LINE_AA
        )
        cv2.imshow('Arabic Sign Language Translator', frame)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam closed.")